# **Word analogy task**

Welcome to your second assignment of this series!

Because word embeddings are very computionally expensive to train, most ML practitioners will load a pre-trained set of embeddings.

**After this assignment you will be able to:**

- Load pre-trained word vectors, and measure similarity using cosine similarity
- Use word embeddings to solve word analogy problems such as Man is to Woman as King is to ______.
- Modify word embeddings to reduce their gender bias (optional)

Let's get started! Run the following cell to load the packages you will need.

In [1]:
# if u got import errors in google colab run this cell for fix!
'''
!pip install numpy==1.24.3
import os
os.kill(os.getpid(), 9)  # Restart the runtime to apply changes
'''

'\n!pip install numpy==1.24.3\nimport os\nos.kill(os.getpid(), 9)  # Restart the runtime to apply changes\n'

In [2]:
!pip install gensim huggingface_hub

In [3]:
import numpy as np
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download

c:\Users\niush\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\niush\anaconda3\lib\site-packages\numpy\.libs\libopenblas.XWYDX2IKJW2NMTWSFYNGFUWKQU3LYTCZ.gfortran-win_amd64.dll
c:\Users\niush\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


Next, lets load the word vectors. For this assignment, we will use 300-dimensional word vectors trained on approximately 100 billion words from the Google News dataset to represent words. (u can use also 50-dimensional GloVe or etc...)

in the following cell, load the model from hugging face hub:
https://huggingface.co/fse/word2vec-google-news-300/tree/main

In [4]:
# Path to the model file
model_path = "./Models/HuggingFace-GoogleNews/word2vec-google-news-300.model"

# Load the Gensim model
word_vectors = KeyedVectors.load(model_path)

# Create a dictionary mapping each word to its corresponding vector
word_to_vec_map = {word: word_vectors[word] for word in word_vectors.key_to_index}


In [5]:
from tabulate import tabulate

# Show 10 words from the vocabulary in a formatted table
print("Sample words:")
sample_words = [[word_vectors.index_to_key[i]] for i in range(10)]
print(tabulate(sample_words, tablefmt="grid"))


Sample words:
+------+
| </s> |
+------+
| in   |
+------+
| for  |
+------+
| that |
+------+
| is   |
+------+
| on   |
+------+
| ##   |
+------+
| The  |
+------+
| with |
+------+
| said |
+------+


**sample Expected Output**:

<table>
    <tr>
        <td>
            in
        </td>
    </tr>
        <tr>
        <td>
            for
        </td>
    </tr>
        <tr>
        <td>
            that
        </td>
    </tr>
        <tr>
        <td>
            is
        </td>
    </tr>
        <tr>
        <td>
            on
        </td>
    </tr>
        <tr>
        <td>
            ##
        </td>
    </tr>
        <tr>
        <td>
            the
        </td>
    </tr>
        <tr>
        <td>
            with
        </td>
    </tr>
        <tr>
        <td>
            said
        </td>
    </tr>
</table>

In [6]:
# Show 10 word vectors and their lengths in a formatted table
print("\nSample word vectors:")

table_data = []
count = 0
for word in word_to_vec_map:
    if count >= 10:
        break
    vector = word_to_vec_map[word]
    short_vec = ", ".join(f"{v:.3f}" for v in vector[:5])  # first 5 numbers, rounded
    table_data.append([word, f"[{short_vec}, ...]", len(vector)])
    count += 1

# Define column headers
headers = ["Word", "Vector (first 5 values)", "Length"]

# Print as a table
print(tabulate(table_data, headers=headers, tablefmt="grid"))



Sample word vectors:
+--------+----------------------------------------------+----------+
| Word   | Vector (first 5 values)                      |   Length |
+========+==============================================+==========+
| </s>   | [0.001, -0.001, 0.000, 0.002, 0.001, ...]    |      300 |
+--------+----------------------------------------------+----------+
| in     | [0.070, 0.087, 0.088, 0.062, 0.069, ...]     |      300 |
+--------+----------------------------------------------+----------+
| for    | [-0.012, -0.047, 0.045, 0.063, -0.018, ...]  |      300 |
+--------+----------------------------------------------+----------+
| that   | [-0.016, -0.028, 0.083, 0.050, -0.110, ...]  |      300 |
+--------+----------------------------------------------+----------+
| is     | [0.007, -0.073, 0.172, 0.023, -0.133, ...]   |      300 |
+--------+----------------------------------------------+----------+
| on     | [0.027, -0.091, 0.028, 0.204, 0.006, ...]    |      300 |
+--------+--

**sample Expected output:**

Sample word vectors:

/s> : [ 0.00112915 -0.00089645  0.00031853  0.00153351  0.00110626]... (len=300)

in : [0.0703125  0.08691406 0.08789062 0.0625     0.06933594]... (len=300)

for : [-0.01177979 -0.04736328  0.04467773  0.06347656 -0.01818848]... (len=300)

that : [-0.01574707 -0.02832031  0.08349609  0.05029297 -0.11035156]... (len=300)

is : [ 0.00704956 -0.07324219  0.171875    0.02258301 -0.1328125 ]... (len=300)

on : [ 0.0267334  -0.09082031  0.02783203  0.20410156  0.00622559]... (len=300)

apple : [ 0.16894531 -0.06298828 -0.00026512  0.06347656  0.08740234]... (len=300)

The : [-0.17285156  0.27929688  0.10693359 -0.15820312 -0.08447266]... (len=300)

with : [-0.02490234  0.02197266 -0.03540039  0.13671875  0.01635742]... (len=300)

said : [-0.00909424 -0.04418945  0.09960938 -0.07617188 -0.05664062]... (len=300)

You've loaded:
- `words`: set of words in the vocabulary.
- `word_to_vec_map`: dictionary mapping words to their google news vector representation.

__You've seen that one-hot vectors do not do a good job capturing what words are similar__. google news or GloVe vectors provide much more useful information about the meaning of individual words. Lets now see how you can use these vectors to decide how similar two words are.



# 1 - Cosine similarity


In [7]:
# GRADED FUNCTION: cosine_similarity

def cosine_similarity(u, v):

    dot = np.dot(u, v)                     # u . v
    
    norm_u = np.linalg.norm(u)            # ||u||
    norm_v = np.linalg.norm(v)            # ||v||
    
    cosine_similar = dot / (norm_u * norm_v)
    

    return cosine_similar

In [8]:
father = word_to_vec_map["father"]
mother = word_to_vec_map["mother"]
ball = word_to_vec_map["ball"]
crocodile = word_to_vec_map["crocodile"]
big = word_to_vec_map["big"]
bigger = word_to_vec_map["bigger"]
small = word_to_vec_map["small"]
smaller = word_to_vec_map["smaller"]

print("cosine_similarity(father, mother) = ", cosine_similarity(father, mother))
print("cosine_similarity(ball, crocodile) = ",cosine_similarity(ball, crocodile))
print("cosine_similarity(big - bigger, small - smaller) = ",cosine_similarity(big - bigger, small - smaller))

cosine_similarity(father, mother) =  0.79014826
cosine_similarity(ball, crocodile) =  0.10283585
cosine_similarity(big - bigger, small - smaller) =  0.5752201


**Expected Output**:

<table>
    <tr>
        <td>
            **cosine_similarity(father, mother)** =
        </td>
        <td>
         0.79014826 (a high number)
        </td>
    </tr>
        <tr>
        <td>
            **cosine_similarity(ball, crocodile)** =
        </td>
        <td>
         0.10283584 (a small number)
        </td>
    </tr>
        <tr>
        <td>
            **cosine_similarity(big - bigger, small - smaller)** =
        </td>
        <td>
         -0.6030071 (high negative number)
        </td>
    </tr>
</table>

In [9]:
def analyze_difference(w1, w2, w3, w4, word_vectors):
    diff1 = word_vectors[w1] - word_vectors[w2]
    diff2 = word_vectors[w3] - word_vectors[w4]
    
    cos_sim = cosine_similarity(diff1, diff2)
    angle_rad = np.arccos(np.clip(cos_sim, -1.0, 1.0))
    angle_deg = np.degrees(angle_rad)
    norm1 = np.linalg.norm(diff1)
    norm2 = np.linalg.norm(diff2)
    
    print(f"{w1}-{w2} vs {w3}-{w4}")
    print(f"  Cosine similarity: {cos_sim:.4f}")
    print(f"  Angle between: {angle_deg:.2f} degrees")
    print(f"  Norms: {norm1:.2f} vs {norm2:.2f}")
    print("-" * 40)

In [10]:
analyze_difference("big", "bigger", "small", "smaller", word_vectors)

big-bigger vs small-smaller
  Cosine similarity: 0.5752
  Angle between: 54.88 degrees
  Norms: 1.96 vs 2.00
----------------------------------------


After you get the correct expected output, please feel free to modify the inputs and measure the cosine similarity between other pairs of words! Playing around the cosine similarity of other inputs will give you a better sense of how word vectors behave.

## 2 - Word analogy task

In the word analogy task, we complete the sentence <font color='brown'>"*a* is to *b* as *c* is to **____**"</font>. An example is <font color='brown'> '*man* is to *woman* as *king* is to *queen*' </font>. In detail, we are trying to find a word *d*, such that the associated word vectors $e_a, e_b, e_c, e_d$ are related in the following manner: $e_b - e_a \approx e_d - e_c$. We will measure the similarity between $e_b - e_a$ and $e_d - e_c$ using cosine similarity.

**Exercise**: Complete the code below to be able to perform word analogies!

In [11]:
# GRADED FUNCTION: complete_analogy
def complete_analogy(word_a, word_b, word_c, word_to_vec_map):
    """
    Performs the word analogy task: a is to b as c is to ____.
    """

    # convert words to lower case
    word_a, word_b, word_c = word_a.lower(), word_b.lower(), word_c.lower()

    # Get the word embeddings e_a, e_b and e_c
    e_a, e_b, e_c = word_to_vec_map[word_a], word_to_vec_map[word_b], word_to_vec_map[word_c]

    words = word_to_vec_map.keys()
    max_cosine_sim = -100
    best_word = None

    # loop over the whole word vector set
    for w in words:
        if w in [word_a, word_b, word_c]:
            continue

        # Compute cosine similarity
        cosine_sim = cosine_similarity(e_b - e_a, word_to_vec_map[w] - e_c)

        if cosine_sim > max_cosine_sim:
            max_cosine_sim = cosine_sim
            best_word = w

    return best_word


Run the cell below to test your code, this may take few minutes.

In [12]:
triads_to_try = [('france', 'paris', 'germany'), ('man', 'woman', 'boy'), ('small', 'smaller', 'large'),
                 ('walk', 'walking', 'swim'), ('he', 'him', 'she'),('bad', 'worse', 'good')]
for triad in triads_to_try:
    print ('{} -> {} :: {} -> {}'.format( *triad, complete_analogy(*triad,word_to_vec_map)))

france -> paris :: germany -> plaster
man -> woman :: boy -> girl
small -> smaller :: large -> larger
walk -> walking :: swim -> swimming
he -> him :: she -> her
bad -> worse :: good -> better


**Expected Output**:

<table>
    <tr>
        <td>
            **france -> paris** ::
        </td>
        <td>
         germany -> berlin
        </td>
        <tr>
        <td>
            **man -> woman ** ::
        </td>
        <td>
         boy -> girl
        </td>
    </tr>
    <tr>
        <td>
            **small -> smaller ** ::
        </td>
        <td>
         large -> larger
        </td>
    </tr>
        <td>
            **walk -> walking ** ::
        </td>
        <td>
         swim -> swimming
        </td>
    </tr>
    <tr>
        <td>
            **he -> him ** ::
        </td>
        <td>
         she -> her
        </td>
    </tr>
    <tr>
        <td>
            **bad -> worse ** ::
        </td>
        <td>
         good -> better
        </td>
    </tr>
</table>

In [13]:
triads_to_try = [('king', 'queen', 'man'), ]
for triad in triads_to_try:
    print ('{} -> {} :: {} -> {}'.format( *triad, complete_analogy(*triad,word_to_vec_map)))

king -> queen :: man -> woman


**Expected Output**:

<table>
    <tr>
        <td>
            king -> queen ::
        </td>
        <td>
            man -> woman
        </td>
        <tr>
</table>

Once you get the correct expected output, please feel free to modify the input cells above to test your own analogies. Try to find some other analogy pairs that do work, but also find some where the algorithm doesn't give the right answer

### Congratulations!

You've come to the end of this assignment. Here are the main points you should remember:

- Cosine similarity a good way to compare similarity between pairs of word vectors. (Though L2 distance works too.)
- For NLP applications, using a pre-trained set of word vectors from the internet is often a good way to get started.

Even though you have finished the graded portions, we recommend you take a look too at the rest of this notebook.

Congratulations on finishing the graded portions of this notebook!


## 3 - Debiasing word vectors (OPTIONAL/UNGRADED)

In the following exercise, you will examine gender biases that can be reflected in a word embedding, and explore algorithms for reducing the bias. In addition to learning about the topic of debiasing, this exercise will also help hone your intuition about what word vectors are doing. This section involves a bit of linear algebra, though you can probably complete it even without being expert in linear algebra, and we encourage you to give it a shot. This portion of the notebook is optional and is not graded.

Lets first see how the GloVe word embeddings relate to gender. You will first compute a vector $g = e_{woman}-e_{man}$, where $e_{woman}$ represents the word vector corresponding to the word *woman*, and $e_{man}$ corresponds to the word vector corresponding to the word *man*. The resulting vector $g$ roughly encodes the concept of "gender". (You might get a more accurate representation if you compute $g_1 = e_{mother}-e_{father}$, $g_2 = e_{girl}-e_{boy}$, etc. and average over them. But just using $e_{woman}-e_{man}$ will give good enough results for now.)


In [14]:
g = word_to_vec_map['woman'] - word_to_vec_map['man']
print(g)

[-8.30078125e-02 -2.08007812e-01 -1.37695312e-01 -2.44140625e-02
  2.83203125e-02 -6.61621094e-02  8.39843750e-02  1.87377930e-02
 -3.17382812e-02  4.66384888e-02 -1.25976562e-01 -9.37500000e-02
  2.09960938e-02  2.63671875e-02  7.49511719e-02  5.34667969e-02
 -8.98437500e-02 -1.12792969e-01  3.27148438e-02  7.00683594e-02
  1.01806641e-01 -4.93927002e-02 -8.12988281e-02 -1.36718750e-01
 -5.38330078e-02 -4.10156250e-02 -9.27734375e-02  3.90625000e-03
 -2.53906250e-02 -1.66503906e-01 -1.85546875e-02 -1.42944336e-01
 -1.61621094e-01 -8.48388672e-02 -1.95312500e-02 -1.30859375e-01
  5.85937500e-02  7.81250000e-02 -1.67480469e-01 -1.21093750e-01
 -1.88720703e-01  1.04980469e-02 -5.37109375e-02 -1.61743164e-02
 -9.27734375e-02  4.49218750e-02  0.00000000e+00  2.49023438e-02
  2.11181641e-02  1.14257812e-01  1.86279297e-01 -1.23390198e-01
 -2.44140625e-04  2.22167969e-01 -1.58691406e-01  1.31835938e-02
 -1.46484375e-02 -2.68554688e-02  1.22070312e-01  2.62451172e-02
  1.24023438e-01 -1.30859

Now, you will consider the cosine similarity of different words with $g$. Consider what a positive value of similarity means vs a negative cosine similarity.

In [15]:
print ('List of names and their similarities with constructed vector:')

# girls and boys name
name_list = ['john', 'marie', 'sophie', 'ronaldo', 'priya', 'rahul', 'danielle', 'reza', 'katy', 'yasmin']

for w in name_list:
    print (w, cosine_similarity(word_to_vec_map[w], g))

List of names and their similarities with constructed vector:
john -0.03143504
marie 0.17699815
sophie 0.1283398
ronaldo -0.09830637
priya 0.09364402
rahul -0.036300182
danielle 0.10128634
reza -0.056654923
katy 0.09632665
yasmin 0.19511871


As you can see, female first names tend to have a positive cosine similarity with our constructed vector $g$, while male first names tend to have a negative cosine similarity. This is not suprising, and the result seems acceptable.

But let's try with some other words.

In [16]:
print('Other words and their similarities:')
word_list = ['lipstick', 'guns', 'science', 'arts', 'literature', 'warrior','doctor', 'tree', 'receptionist',
             'technology',  'fashion', 'teacher', 'engineer', 'pilot', 'computer', 'singer']
for w in word_list:
    print (w, cosine_similarity(word_to_vec_map[w], g))

Other words and their similarities:
lipstick 0.26272777
guns -0.12910882
science 0.038018893
arts 0.13724276
literature 0.18530814
warrior -0.09518411
doctor 0.16264015
tree 0.063181736
receptionist 0.3256916
technology 0.06544881
fashion 0.16653466
teacher 0.14762524
engineer -0.057295308
pilot 0.036814738
computer 0.09310092
singer 0.15405332


### 3.1 - Neutralize bias for non-gender specific words

In [17]:
def neutralize(word, g, word_to_vec_map):
    """
    Removes the bias of "word" by projecting it on the space orthogonal to the bias axis.
    This function ensures that gender neutral words are zero in the gender subspace.

    Arguments:
        word -- string indicating the word to debias
        g -- numpy-array of shape (50,), corresponding to the bias axis (such as gender)
        word_to_vec_map -- dictionary mapping words to their corresponding vectors.

    Returns:
        e_debiased -- neutralized word vector representation of the input "word"
    """
    e = word_to_vec_map[word]
    e_biascomponent = np.dot(e, g) / np.linalg.norm(g)**2 * g
    e_debiased = e - e_biascomponent

    return e_debiased

In [18]:
e = "receptionist"
print("cosine similarity between " + e + " and g, before neutralizing: ", cosine_similarity(word_to_vec_map["receptionist"], g))

e_debiased = neutralize("receptionist", g, word_to_vec_map)
print("cosine similarity between " + e + " and g, after neutralizing: ", cosine_similarity(e_debiased, g))

cosine similarity between receptionist and g, before neutralizing:  0.3256916
cosine similarity between receptionist and g, after neutralizing:  1.1497245e-08


**Expected Output**: The second result is essentially 0, up to numerical roundof (on the order of $10^{-17}$).


<table>
    <tr>
        <td>
            **cosine similarity between receptionist and g, before neutralizing:** :
        </td>
        <td>
         0.330779417506
        </td>
    </tr>
        <tr>
        <td>
            **cosine similarity between receptionist and g, after neutralizing:** :
        </td>
        <td>
         -3.26732746085e-17
    </tr>
</table>

### 3.2 - Equalization algorithm for gender-specific words

In [19]:
def equalize(pair, bias_axis, word_to_vec_map):
    
    ### START CODE HERE ###
    w1, w2 = pair
    e_w1, e_w2 = word_to_vec_map[w1], word_to_vec_map[w2]

    # Step 2: Compute the mean vector
    mu = (e_w1 + e_w2) / 2

    # Step 3: Compute projections
    mu_B = np.dot(mu, bias_axis) / np.linalg.norm(bias_axis)**2 * bias_axis
    mu_perp = mu - mu_B

    # Step 4: Compute bias components of e_w1 and e_w2
    e_w1B = np.dot(e_w1, bias_axis) / np.linalg.norm(bias_axis)**2 * bias_axis
    e_w2B = np.dot(e_w2, bias_axis) / np.linalg.norm(bias_axis)**2 * bias_axis

    # Step 5: Correct the bias components
    corrected_e_w1B = np.sqrt(abs(1 - np.linalg.norm(mu_perp)**2)) * (e_w1B - mu_B) / np.linalg.norm((e_w1 - mu_perp) - mu_B)
    corrected_e_w2B = np.sqrt(abs(1 - np.linalg.norm(mu_perp)**2)) * (e_w2B - mu_B) / np.linalg.norm((e_w2 - mu_perp) - mu_B)

    # Step 6: Combine corrected bias components with orthogonal part
    e1 = corrected_e_w1B + mu_perp
    e2 = corrected_e_w2B + mu_perp
    ### END CODE HERE ###

    return e1, e2


In [20]:
print("cosine similarities before equalizing:")
print("cosine_similarity(word_to_vec_map[\"man\"], gender) = ", cosine_similarity(word_to_vec_map["man"], g))
print("cosine_similarity(word_to_vec_map[\"woman\"], gender) = ", cosine_similarity(word_to_vec_map["woman"], g))
print()
e1, e2 = equalize(("man", "woman"), g, word_to_vec_map)
print("cosine similarities after equalizing:")
print("cosine_similarity(e1, gender) = ", cosine_similarity(e1, g))
print("cosine_similarity(e2, gender) = ", cosine_similarity(e2, g))

cosine similarities before equalizing:
cosine_similarity(word_to_vec_map["man"], gender) =  -0.15934694
cosine_similarity(word_to_vec_map["woman"], gender) =  0.5120308

cosine similarities after equalizing:
cosine_similarity(e1, gender) =  -0.66846603
cosine_similarity(e2, gender) =  0.668466


**Expected Output**:

cosine similarities before equalizing:
<table>
    <tr>
        <td>
            **cosine_similarity(word_to_vec_map["man"], gender)** =
        </td>
        <td>
         -0.117110957653
        </td>
    </tr>
        <tr>
        <td>
            **cosine_similarity(word_to_vec_map["woman"], gender)** =
        </td>
        <td>
         0.356666188463
        </td>
    </tr>
</table>

cosine similarities after equalizing:
<table>
    <tr>
        <td>
            **cosine_similarity(u1, gender)** =
        </td>
        <td>
         -0.700436428931
        </td>
    </tr>
        <tr>
        <td>
            **cosine_similarity(u2, gender)** =
        </td>
        <td>
         0.700436428931
        </td>
    </tr>
</table>

Please feel free to play with the input words in the cell above, to apply equalization to other pairs of words.

These debiasing algorithms are very helpful for reducing bias, but are not perfect and do not eliminate all traces of bias. For example, one weakness of this implementation was that the bias direction $g$ was defined using only the pair of words _woman_ and _man_. As discussed earlier, if $g$ were defined by computing $g_1 = e_{woman} - e_{man}$; $g_2 = e_{mother} - e_{father}$; $g_3 = e_{girl} - e_{boy}$; and so on and averaging over them, you would obtain a better estimate of the "gender" dimension in the 50 dimensional word embedding space. Feel free to play with such variants as well.  
                     

Another Examples:

In [21]:

print("==== Testing neutralize() on neutral words ====\n")
neutral_words = ["scientist", "nurse", "engineer"]

for word in neutral_words:
    before = cosine_similarity(word_to_vec_map[word], g)
    e_debiased = neutralize(word, g, word_to_vec_map)
    after = cosine_similarity(e_debiased, g)
    print(f"Word: {word}")
    print(f"  Cosine similarity before neutralizing: {before:.6f}")
    print(f"  Cosine similarity after neutralizing:  {after:.6f}")
    print("---")

print("\n==== Testing equalize() on gendered word pairs ====\n")
equalize_pairs = [("king", "queen"), ("brother", "sister")]

for w1, w2 in equalize_pairs:
    before_1 = cosine_similarity(word_to_vec_map[w1], g)
    before_2 = cosine_similarity(word_to_vec_map[w2], g)
    e1, e2 = equalize((w1, w2), g, word_to_vec_map)
    after_1 = cosine_similarity(e1, g)
    after_2 = cosine_similarity(e2, g)
    print(f"Pair: ({w1}, {w2})")
    print(f"  Cosine similarity before equalizing: {w1} = {before_1:.6f}, {w2} = {before_2:.6f}")
    print(f"  Cosine similarity after equalizing:  {w1} = {after_1:.6f}, {w2} = {after_2:.6f}")
    print("---")


==== Testing neutralize() on neutral words ====

Word: scientist
  Cosine similarity before neutralizing: 0.026401
  Cosine similarity after neutralizing:  -0.000000
---
Word: nurse
  Cosine similarity before neutralizing: 0.337688
  Cosine similarity after neutralizing:  -0.000000
---
Word: engineer
  Cosine similarity before neutralizing: -0.057295
  Cosine similarity after neutralizing:  0.000000
---

==== Testing equalize() on gendered word pairs ====

Pair: (king, queen)
  Cosine similarity before equalizing: king = -0.109334, queen = 0.263174
  Cosine similarity after equalizing:  king = -0.384879, queen = 0.384879
---
Pair: (brother, sister)
  Cosine similarity before equalizing: brother = -0.113682, sister = 0.316114
  Cosine similarity after equalizing:  brother = -0.459553, sister = 0.459553
---


### Congratulations

You have come to the end of this notebook, and have seen a lot of the ways that word vectors can be used as well as  modified.

Congratulations on finishing this notebook!


**References**:
- The debiasing algorithm is from Bolukbasi et al., 2016, [Man is to Computer Programmer as Woman is to
Homemaker? Debiasing Word Embeddings](https://papers.nips.cc/paper/6228-man-is-to-computer-programmer-as-woman-is-to-homemaker-debiasing-word-embeddings.pdf)
- The GloVe word embeddings were due to Jeffrey Pennington, Richard Socher, and Christopher D. Manning. (https://nlp.stanford.edu/projects/glove/)
